In [9]:
import matplotlib
matplotlib.use("QtAgg")  

In [10]:
import numpy as np
import mne
mne.set_log_level("INFO")  # INFO

In [11]:
set_path = r"./00002.set"
raw = mne.io.read_raw_eeglab(set_path, preload=True)
raw.info

Reading C:\Users\lapto\Desktop\eeg_tangjiabao\Dataset\00002.fdt
Reading 0 ... 1330312  =      0.000 ...  5321.248 secs...


<Info | 8 non-empty values
 bads: []
 ch_names: FP1, FPz, FP2, AF7, AF3, AF4, AF8, F7, F5, F3, F1, Fz, F2, F4, ...
 chs: 65 EEG
 custom_ref_applied: False
 dig: 65 items (65 EEG)
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 65
 projs: []
 sfreq: 250.0 Hz
>

In [12]:
print(raw)
print("时长(s):", raw.times[-1])
print("采样率(Hz):", raw.info["sfreq"])
print("通道数:", len(raw.ch_names))

<RawEEGLAB | 00002.fdt, 65 x 1330313 (5321.3 s), ~659.8 MiB, data loaded>
时长(s): 5321.248
采样率(Hz): 250.0
通道数: 65


In [13]:
raw.plot(n_channels=32, duration=10, scalings="auto")

<MNEBrowseFigure size 1000x1000 with 4 Axes>

In [14]:
import numpy as np
import mne

# 原始标注概况（
ann0 = raw.annotations
desc0 = np.array(ann0.description)
print("原始 annotations 总数:", len(ann0))
for d in np.unique(desc0):
    print(f"{d}: {np.sum(desc0==d)}")

# 删除结束标记,每对里的第2个
ann = raw.annotations
onset = ann.onset
duration = ann.duration
desc = np.array(ann.description)

keep = np.ones(len(ann), dtype=bool)

# 排除boundary/BAD
def is_pair_target(d: str) -> bool:
    d_low = d.lower()
    if d_low.startswith("bad") or "boundary" in d_low:
        return False
    return True

for d in np.unique(desc):
    if not is_pair_target(d):
        continue

    idx = np.where(desc == d)[0]
    idx = idx[np.argsort(onset[idx])]

    # 次数 <2 不动
    if len(idx) < 2:
        continue

    # 若数量为奇数,直接丢掉最后一个，避免它被当成开始
    if len(idx) % 2 == 1:
        keep[idx[-1]] = False
        idx = idx[:-1]

    # 删除每对里的第2个
    keep[idx[1::2]] = False

new_ann = mne.Annotations(
    onset=onset[keep],
    duration=duration[keep],
    description=desc[keep],
    orig_time=ann.orig_time
)
raw.set_annotations(new_ann)

# 删除后的标注概况
ann1 = raw.annotations
desc1 = np.array(ann1.description)
print("\n删除结束标记后 annotations 总数:", len(ann1))
for d in np.unique(desc1):
    print(f"{d}: {np.sum(desc1==d)}")

# 重新画一遍 raw
raw.plot(n_channels=32, duration=10, scalings="auto")


原始 annotations 总数: 556
1: 144
2: 150
3: 132
4: 130

删除结束标记后 annotations 总数: 278
1: 72
2: 75
3: 66
4: 65


<MNEBrowseFigure size 1000x1000 with 4 Axes>

In [15]:
import mne
out_set = "./00002_modified.set"
# 修改后的 raw 导出为 EEGLAB .set
mne.export.export_raw(out_set, raw, fmt="eeglab", overwrite=True)
print("Saved:", out_set)

Overwriting existing file.
Saved: ./00002_modified.set


In [16]:
raw.set_channel_types({"DIGITAL IN": "stim"})
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, match_case=False)

<RawEEGLAB | 00002.fdt, 65 x 1330313 (5321.3 s), ~659.8 MiB, data loaded>

In [17]:
print(raw.annotations)

<Annotations | 278 segments: 1 (72), 2 (75), 3 (66), 4 (65)>


In [18]:
# 画传感器布局
raw.plot_sensors(show_names=True, show=True)

<Figure size 800x800 with 1 Axes>

In [19]:
# 做平均参考
raw.set_eeg_reference("average", projection=False)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


<RawEEGLAB | 00002.fdt, 65 x 1330313 (5321.3 s), ~659.8 MiB, data loaded>

In [20]:
#　频谱图 PSD　检查噪声/工频
raw.compute_psd(fmin=0.5, fmax=50).plot()

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


<MNELineFigure size 1250x437.5 with 2 Axes>

In [21]:
# 复制一份用于对比
raw_filt = raw.copy()

In [22]:
raw_filt.notch_filter(
    freqs=[48, 52],
    picks="eeg",
    method="fir",         
    phase="zero"           # 零相位，避免相位畸变
)

# 2) Band-pass (MI 常用 7–30 Hz)
raw_filt.filter(
    l_freq=0.1, h_freq=40.0,
    picks="eeg",
    method="fir",
    phase="zero",
    fir_design="firwin"
)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 1651 samples (6.604 s)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 8251 samples (33.004 s)



<RawEEGLAB | 00002.fdt, 65 x 1330313 (5321.3 s), ~659.8 MiB, data loaded>

In [23]:
# PSD 对比
raw.compute_psd(fmin=0.5, fmax=50).plot()
raw_filt.compute_psd(fmin=0.5, fmax=50).plot()

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).
Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


<MNELineFigure size 1250x437.5 with 2 Axes>

In [24]:
# # 找坏道
# raw_filt.plot(n_channels=32, duration=10, scalings="auto")

In [25]:
# # 如果发现明显飘、爆高噪声的通道，加入 bads
# raw_filt.info["bads"] = ["TP7", "CP1"]
# # 并做插值（可选，坏道少时很常用）
# raw_filt.interpolate_bads(reset_bads=True)

In [26]:
# # 拟合 ICA
# rank = mne.compute_rank(raw_filt)
# print(rank)
# 
# ica = mne.preprocessing.ICA(
#     n_components=rank['eeg'],
#     random_state=97,
#     method="fastica"
# )
# ica.fit(raw_filt, picks="eeg")

In [27]:
ica = mne.preprocessing.ICA(
    n_components=32,
    random_state=97,
    method="fastica"
)

ica.fit(raw_filt, picks="eeg")

Fitting ICA to data using 64 channels (please be patient, this may take a while)
Selecting by number: 32 components
Fitting ICA took 87.2s.


Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,86 iterations on raw data (1330313 samples)
ICA components,32
Available PCA components,64
Channel types,eeg
ICA components marked for exclusion,—


In [28]:
# 看 ICA 成分（成分拓扑）
ica.plot_components()

Channels marked as bad:
none
Attempting to create new mne-python configuration file:
C:\Users\lapto\.mne\mne-python.json
Could not read the C:\Users\lapto\.mne\mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
Channels marked as bad:
none
Channels marked as bad:
none
Channels marked as bad:
none


KeyboardInterrupt: 

In [ ]:
# # 看成分时间序列
# ica.plot_properties(raw_filt, picks=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19])  

In [30]:
# 导出 ICA 供 realtime_eeg_classification 加载（与训练同一分解，含 exclude；无需在线重采拟合）
import os
ica_training_path = os.path.join("Dataset", "ica_training.fif")
os.makedirs(os.path.dirname(ica_training_path) or ".", exist_ok=True)
ica.save(ica_training_path)
print(f"已导出训练 ICA -> {os.path.abspath(ica_training_path)}")

Writing ICA solution to C:\Users\lapto\Desktop\eeg_tangjiabao\Dataset\Dataset\ica_training.fif...
已导出训练 ICA -> C:\Users\lapto\Desktop\eeg_tangjiabao\Dataset\Dataset\ica_training.fif


C:\Users\lapto\AppData\Local\Temp\ipykernel_16588\3419805458.py:5: RuntimeWarning: This filename (Dataset\ica_training.fif) does not conform to MNE naming conventions. All ICA files should end with -ica.fif, -ica.fif.gz, _ica.fif or _ica.fif.gz
  ica.save(ica_training_path)


In [ ]:
# 再画 PSD 看看改善
raw_ica.compute_psd(fmin=0.5, fmax=50).plot()

In [31]:
# 读取事件 events
events, event_id = mne.events_from_annotations(raw_ica)
print("event_id:", event_id)
print("events.shape:", events.shape)

Used Annotations descriptions: [np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4')]
event_id: {np.str_('1'): 1, np.str_('2'): 2, np.str_('3'): 3, np.str_('4'): 4}
events.shape: (278, 3)


In [32]:
tmin, tmax = -0.2, 3.0   # 先按 0-6s
baseline = -0.2, 0   # 如果 tmin=0 就不做 baseline；想做就改成 (-0.2, 0)

epochs = mne.Epochs(
    raw_ica,
    events=events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    baseline=baseline,
    preload=True,
    picks="eeg",
    reject_by_annotation=True
)
print(epochs)

Not setting metadata
278 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 278 events and 801 original time points ...
0 bad epochs dropped
<Epochs | 278 events (all good), -0.2 – 3 s (baseline -0.2 – 0 s), ~108.8 MiB, data loaded,
 np.str_('1'): 72
 np.str_('2'): 75
 np.str_('3'): 66
 np.str_('4'): 65>


In [33]:
# # Epochs 浏览
epochs.plot(n_channels=32, scalings="auto")

<MNEBrowseFigure size 1000x1000 with 4 Axes>

In [34]:
# # 每个动作的平均波形 Evoked （ERP / ERD 的基础）
evokeds = {cond: epochs[cond].average() for cond in event_id.keys()}
list(evokeds.keys())

[np.str_('1'), np.str_('2'), np.str_('3'), np.str_('4')]

In [ ]:
# # 画每个条件的全通道波形
# for cond, evo in evokeds.items():
#     evo.plot(window_title=cond, spatial_colors=True)

In [ ]:
# # 指定几个通道对比
# # 看 C3/C4/Cz（运动相关常用）
# mne.viz.plot_compare_evokeds(evokeds, picks=["C3", "Cz", "C4"])

In [ ]:
# # ERP 拓扑演化
# for cond, evo in evokeds.items():
#     evo.plot_topomap(times=np.linspace(0.6, 3, 8), ch_type="eeg", time_unit="s")

In [ ]:
# # 时频图（ERSP / ERD-ERS 常用）
# freqs = np.linspace(4, 40, 30)
# n_cycles = freqs / 2.0
# 
# cond0 = list(event_id.keys())[0]  # 先拿第一个条件示例
# power = mne.time_frequency.tfr_morlet(
#     epochs[cond0],
#     freqs=freqs,
#     n_cycles=n_cycles,
#     return_itc=False,
#     average=True
# )
# 
# power.plot(picks=["C3"], baseline=None, mode=None, title=f"TFR - {cond0} - C3")

In [ ]:
# # 拓扑
# power.plot_topomap(tmin=1.0, tmax=3.0, fmin=8, fmax=30, baseline=None, mode=None)

In [ ]:
# # 质量检测
# # 通道协方差/噪声结构
# cov = mne.compute_raw_covariance(raw_ica, picks="eeg")
# mne.viz.plot_cov(cov, raw_ica.info)
# # GFP/全局场功率
# for cond, evo in evokeds.items():
#     evo.plot(gfp=True, titles=dict(eeg=f"GFP - {cond}"))

In [ ]:
# ———直接把 每个 trial 的 EEG 段导出为 .npz，供后续深度学习（PyTorch / TF）使用。———

In [35]:
sfreq = epochs.info["sfreq"]
n_times = len(epochs.times)
print("sfreq =", sfreq, "Hz")
print("n_times =", n_times, "samples")
print("epoch长度(s) =", n_times/sfreq)

sfreq = 250.0 Hz
n_times = 801 samples
epoch长度(s) = 3.204


In [36]:
# X：形状默认是 (n_epochs, n_channels, n_times)
X = epochs.get_data()  # numpy array
print("X shape:", X.shape)  # (trial, ch, time)

X shape: (278, 64, 801)


In [37]:
# MNE 的 epoch 顺序与 epochs.events 对齐
y = epochs.events[:, 2].copy()   # 取 event code: 1,2,3,4
print("y unique:", np.unique(y, return_counts=True))
# 转成 0~3 符合常见 DL 分类习惯
y = y - 1
print("y unique (0-based):", np.unique(y, return_counts=True))

y unique: (array([1, 2, 3, 4]), array([72, 75, 66, 65]))
y unique (0-based): (array([0, 1, 2, 3]), array([72, 75, 66, 65]))


In [38]:
# # 假设四类注释名就是 event_id 里的四个 key
# keys = sorted(event_id.keys())               # 或者你自己写成 ['left','right','foot','tongue']
# map_code_to_y = {event_id[k]: i for i,k in enumerate(keys)}
# y = np.array([map_code_to_y[c] for c in epochs.events[:,2]], dtype=np.int64)

In [39]:
# 转 dtype：
X = X.astype(np.float32)
y = y.astype(np.int64)

In [40]:
# 按每个 trial 的每个通道做标准化
eps = 1e-8
mean = X.mean(axis=2, keepdims=True)
std  = X.std(axis=2, keepdims=True)
Xz = (X - mean) / (std + eps)

print("Xz mean/std check:", Xz.mean(), Xz.std())

Xz mean/std check: -2.448529e-10 0.99757475


In [41]:
# 导出为 npz（包含元信息）
out_path = "002.npz"

np.savez_compressed(
    out_path,
    X=Xz,              # (N, C, T) 标准化后的数据
    y=y,               # (N,) 0..3
    sfreq=sfreq,
    ch_names=np.array(epochs.ch_names),
    times=epochs.times.astype(np.float32),
    event_id=event_id,
    tmin=tmin,
    tmax=tmax
)

print("saved:", out_path)

saved: 002.npz


In [42]:
# 验证读取
d = np.load(out_path, allow_pickle=True)
print(d["X"].shape, d["y"].shape)
print("sfreq:", d["sfreq"])
print("classes:", np.unique(d["y"], return_counts=True))
print("ch_names[:5]:", d["ch_names"][:5])

(278, 64, 801) (278,)
sfreq: 250.0
classes: (array([0, 1, 2, 3]), array([72, 75, 66, 65]))
ch_names[:5]: ['FP1' 'FPz' 'FP2' 'AF7' 'AF3']
